In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)
print("Setup complete and ready!")

TensorFlow Version: 2.21.0
Setup complete and ready!


In [4]:
import os
import tensorflow as tf

# Configuration Parameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ["ok_front", "def_front"]

# Correct Folder Paths based on your VS Code Explorer
TRAIN_DIR = "casting_data/casting_data/train"
TEST_DIR = "casting_data/casting_data/test"

# Training Dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=CLASS_NAMES,
    validation_split=0.20,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Validation Dataset
val_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    class_names=CLASS_NAMES,
    validation_split=0.20,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

# Testing Dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# Performance Optimization
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print("Data Pipeline successfully initialized!")

Found 6633 files belonging to 2 classes.
Using 5307 files for training.
Found 6633 files belonging to 2 classes.
Using 1326 files for validation.
Found 715 files belonging to 2 classes.
Data Pipeline successfully initialized!


In [5]:
from tensorflow.keras import layers, models

# 1. Data Augmentation Pipeline
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.10)
], name="data_augmentation")

# 2. Custom CNN Architecture
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    data_augmentation,
    layers.Rescaling(1.0 / 255.0),
    
    # Convolutional Block 1
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    
    # Convolutional Block 2
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    
    # Convolutional Block 3
    layers.Conv2D(128, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(),
    
    # Classification Head
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.40),
    
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    
    # Single Sigmoid Output Neuron for Binary Classification
    layers.Dense(1, activation="sigmoid")
], name="Casting_Defect_CNN")

# 3. Model Compilation
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# Display Model Summary
model.summary()

Model: "Casting_Defect_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,569 (396.75 KB)

 Trainable params: 101,569 (396.75 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
import os

# Create directory to save the trained model
os.makedirs("models", exist_ok=True)

# 1. Configure Regularization Callbacks
callbacks = [
    # Stop training if validation loss doesn't improve for 5 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    
    # Reduce learning rate when learning plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
    
    # Save the best model state
    tf.keras.callbacks.ModelCheckpoint(
        filepath="models/best_casting_defect_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

# 2. Train the Model
EPOCHS = 25

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/25


c:\Users\Kabita Kanchan\OneDrive\Attachments\Pictures\Desktop\casting product image data for quality inspection\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


166/166 ━━━━━━━━━━━━━━━━━━━━ 102s 600ms/step - accuracy: 0.5542 - loss: 0.6899 - precision: 0.5608 - recall: 0.9452 - val_accuracy: 0.5897 - val_loss: 0.6821 - val_precision: 0.5897 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/25
166/166 ━━━━━━━━━━━━━━━━━━━━ 93s 562ms/step - accuracy: 0.5608 - loss: 0.6861 - precision: 0.5608 - recall: 1.0000 - val_accuracy: 0.5897 - val_loss: 0.6778 - val_precision: 0.5897 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/25
166/166 ━━━━━━━━━━━━━━━━━━━━ 93s 558ms/step - accuracy: 0.5615 - loss: 0.6843 - precision: 0.5618 - recall: 0.9909 - val_accuracy: 0.5897 - val_loss: 0.6672 - val_precision: 0.5897 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 4/25
166/166 ━━━━━━━━━━━━━━━━━━━━ 93s 557ms/step - accuracy: 0.5657 - loss: 0.6826 - precision: 0.5693 - recall: 0.9261 - val_accuracy: 0.5897 - val_loss: 0.6576 - val_precision: 0.5897 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 5/25
166/166 ━━━━━━━━━━━━━━━━━━━━ 90s 543ms/step - accur

In [8]:
import os
import tensorflow as tf

def predict_product_quality(image_path, threshold=0.40):
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = tf.expand_dims(tf.keras.utils.img_to_array(img), axis=0)
    
    prob = float(model.predict(img_array, verbose=0)[0][0])
    
    if prob >= threshold:
        status = "DEFECTIVE (1)"
        action = "Send for manual inspection / Reject"
    else:
        status = "NON-DEFECTIVE (0)"
        action = "Product passed inspection"
        
    print("=" * 50)
    print(f"Image File: {image_path}")
    print(f"Prediction: {status}")
    print(f"Defect Probability: {prob:.2%}")
    print(f"Recommended Action: {action}")
    print("=" * 50)

# Automatically pick the first image from test/def_front folder
test_dir = "casting_data/casting_data/test/def_front"
sample_image = os.path.join(test_dir, os.listdir(test_dir)[0])

# Run prediction
predict_product_quality(sample_image)

Image File: casting_data/casting_data/test/def_front\cast_def_0_1059.jpeg
Prediction: DEFECTIVE (1)
Defect Probability: 93.28%
Recommended Action: Send for manual inspection / Reject
